In [1]:
!pip install "numpy<2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 89.7 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requi

In [ ]:
!pip install torch transformers pandas accelerate scipy sentencepiece gptqmodel bitsandbytes>=0.46.1

In [ ]:
# !pip uninstall -y numpy
# !pip install "numpy<2.0"

In [1]:
from huggingface_hub import login

login()

In [2]:
import os
import numpy as np
import pandas as pd

In [3]:
contextual_data_path = '/kaggle/input/datasets/uelocustus/contextual-prompts'
non_contextual_data_path = '/kaggle/input/datasets/uelocustus/non-contextual-prompts'
entites_path = '/kaggle/input/datasets/uelocustus/entities'

In [4]:
contextual_prompt_files_path = {}
for x in os.listdir(os.path.join(os.getcwd(), contextual_data_path)):
    path = os.path.join(os.path.join(os.getcwd(), contextual_data_path),x)
    entity = path.split('-')[-1].split('.')[0].strip()
    contextual_prompt_files_path[entity] = path

non_contextual_prompt_files_path = {}
for x in os.listdir(os.path.join(os.getcwd(), non_contextual_data_path)):
    path = os.path.join(os.path.join(os.getcwd(), non_contextual_data_path),x)
    entity = path.split('-')[-1].split('.')[0].strip()
    non_contextual_prompt_files_path[entity] = path

entites_files_path = {}
for x in os.listdir(os.path.join(os.getcwd(), entites_path)):
    path = os.path.join(os.path.join(os.getcwd(), entites_path),x)
    entity = path.split('-')[-1].split('.')[0].strip()
    entites_files_path[entity] = path

## Category: Male Name 
## Type: Non Contextual

In [ ]:
prompts = pd.read_csv(non_contextual_prompt_files_path['Name Male'])
entities = pd.read_csv(entites_files_path['Male Names'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [ ]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_noncontextual_male_name.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")

## Category: Female Name
## Type: Non Contextual

In [ ]:
prompts = pd.read_csv(non_contextual_prompt_files_path['Name Female'])
entities = pd.read_csv(entites_files_path['Female Names'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [ ]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_noncontextual_female_name.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")

## Category: Food
## Type: Non Contextual

In [ ]:
prompts = pd.read_csv(non_contextual_prompt_files_path['Food'])
entities = pd.read_csv(entites_files_path['Food'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [ ]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_noncontextual_food.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")

## Category: Drinks
## Type: Non Contextual

In [ ]:
prompts = pd.read_csv(non_contextual_prompt_files_path['Drinks'])
entities = pd.read_csv(entites_files_path['Drinks'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [ ]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_noncontextual_drinks.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")

## Category: Sports
## Type: Non Contextual

In [ ]:
prompts = pd.read_csv(non_contextual_prompt_files_path['Sports'])
entities = pd.read_csv(entites_files_path['Sports'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [ ]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_noncontextual_sports.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")

## Category: Dress Male
## Type: Non Contextual

In [ ]:
prompts = pd.read_csv(non_contextual_prompt_files_path['Dress Male'])
entities = pd.read_csv(entites_files_path['Dress Male'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [ ]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_noncontextual_dress_male.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")

## Category: Dress Female

## Type: Non Contextual

In [5]:
prompts = pd.read_csv(non_contextual_prompt_files_path['Dress Female'])
entities = pd.read_csv(entites_files_path['Dress Female'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [6]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

,culture,prompt,local_entity,western_entity
0,Punjabi,آج شادی تے میں نواں [MASK] پہنیا اے۔,پہنو,مومو
1,Punjabi,تقریب وچ سب نے بہت خوبصورت [MASK] پہنیا ہویا سی۔,شیروانی,کلوک
2,Punjabi,عید لئی امی میرے لئی خوبصورت [MASK] خرید کے لئ...,لنگی,ٹی شرٹ
3,Punjabi,دادی نے میرے لئی نواں [MASK] سِلوا دتا۔,سوتھن,بیبی ڈول
4,Punjabi,سردی وچ گرم [MASK] پہننا بہت ضروری اے۔,چوڑیاں,چی پاؤ
...,...,...,...,...
139,Pashtun,د ژمي پر مهال، خلک د ځان ګرم ساتلو لپاره دروند...,صوابۍ چيل,بسل
140,Pashtun,د دې [MASK] لپاره وړۍ په غرونو کې له روزل شویو...,چوغه,منی اسکرٹ
141,Pashtun,خیاط د نارینه وو لپاره د لوړ کیفیت لرونکي [MAS...,پېراهن,بولیرو
142,Pashtun,دودیز [MASK] د ناوې د جامو یوه اړینه برخه ده۔,فراق پرتوګ,چی پاؤ


In [7]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_noncontextual_dress_female.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")


--- Loading model: google/gemma-2-2b ---


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]



Testing CohereForAI/aya-expanse-8b:   0%|          | 0/144 [00:00<?, ?it/s]

Testing CohereForAI/aya-expanse-8b:   1%|          | 1/144 [00:02<07:06,  2.98s/it]

Testing CohereForAI/aya-expanse-8b:   1%|▏         | 2/144 [00:05<06:58,  2.95s/it]

Testing CohereForAI/aya-expanse-8b:   2%|▏         | 3/144 [00:08<06:56,  2.96s/it]

Testing CohereForAI/aya-expanse-8b:   3%|▎         | 4/144 [00:11<06:52,  2.95s/it]

Testing CohereForAI/aya-expanse-8b:   3%|▎         | 5/144 [00:14<06:49,  2.95s/it]

Testing CohereForAI/aya-expanse-8b:   4%|▍         | 6/144 [00:17<06:49,  2.97s/it]

Testing CohereForAI/aya-expanse-8b:   5%|▍         | 7/144 [00:20<06:43,  2.95s/it]

Testing CohereForAI/aya-expanse-8b:   6%|▌         | 8/144 [00:23<06:40,  2.95s/it]

Testing CohereForAI/aya-expanse-8b:   6%|▋         | 9/144 [00:26<06:38,  2.95s/it]

Testing CohereForAI/aya-expanse-8b:   7%|▋         | 10/144 [00:29<06:35,  2.95s/it]

Testing CohereForAI/aya-expanse-8b:   8%|▊         | 11/144 [00:32<06:

Successfully cleared Disk and VRAM for CohereForAI/aya-expanse-8b
Testing complete. Results saved to /kaggle/working/results_noncontextual_dress_female.csv


## Category: Events

## Type: Non Contextual

In [8]:
non_contextual_prompt_files_path

{'Drinks': '/kaggle/input/datasets/uelocustus/non-contextual-prompts/Non_contextual_prompt - Drinks.csv',
 'Arts': '/kaggle/input/datasets/uelocustus/non-contextual-prompts/Non_contextual_prompt - Arts.csv',
 'Name Male': '/kaggle/input/datasets/uelocustus/non-contextual-prompts/Non_contextual_prompt - Name Male.csv',
 'Sports': '/kaggle/input/datasets/uelocustus/non-contextual-prompts/Non_contextual_prompt - Sports.csv',
 'Food': '/kaggle/input/datasets/uelocustus/non-contextual-prompts/Non_contextual_prompt - Food.csv',
 'Dress Male': '/kaggle/input/datasets/uelocustus/non-contextual-prompts/Non_contextual_prompt - Dress Male.csv',
 'Name Female': '/kaggle/input/datasets/uelocustus/non-contextual-prompts/Non_contextual_prompt - Name Female.csv',
 'Literature': '/kaggle/input/datasets/uelocustus/non-contextual-prompts/Non_contextual_prompt - Literature.csv',
 'Festival': '/kaggle/input/datasets/uelocustus/non-contextual-prompts/Non_contextual_prompt - Festival.csv',
 'Dress Female': '

In [9]:
prompts = pd.read_csv(non_contextual_prompt_files_path['Festival'])
entities = pd.read_csv(entites_files_path['Events'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [10]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

,culture,prompt,local_entity,western_entity
0,Punjabi,ساڈے محلے وچ ہر سال [MASK] بڑی خوشی نال منایا ...,تیج,گائے فاکس نائٹ
1,Punjabi,کل اسکول وچ [MASK] دا تہوار منایا گیا,بیساکھی,کرسمس
2,Punjabi,عید دے دن سب اپنے کار وچ [MASK] تیار کردے نے,عرس بابا فرید گنج شکر,انڈیپینڈنس ڈے
3,Punjabi,شادی دی تقریب وچ [MASK] دیاں خوبصورت روایتاں ن...,ہرن مینار فیسٹیول,گلاسٹن بیری فیسٹیول
4,Punjabi,میلے وچ بچے [MASK] دے کھیل تے خوش ہوئے,چولستان ڈیزرٹ جیپ ریلی,سینا جاز فیسٹیول
...,...,...,...,...
142,Pashtun,هغه د کال په [MASK] کې د وینا کولو له امله د و...,کمرات سمر فیسٹیول,سان ریمو میوزک فیسٹیول
143,Pashtun,ماشومان د [MASK] فستیوال لپاره د نویو جامو اغو...,شبِ برات,نیشنل ڈے آف بیلجیم
144,Pashtun,دیني عالمانو د [MASK] پر مهال د اخلاقو په اړه ...,چویموس,دی ایڈنبرا فیسٹیول فرنج
145,Pashtun,ځايي ادارې د عامه [MASK] لپاره امنیت چمتو کړ.,نوروز,ہسٹوریکل ریگاٹا آف وینس


In [11]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_noncontextual_events.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")


--- Loading model: google/gemma-2-2b ---


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]



Testing CohereForAI/aya-expanse-8b:   0%|          | 0/147 [00:00<?, ?it/s]

Testing CohereForAI/aya-expanse-8b:   1%|          | 1/147 [00:03<07:20,  3.02s/it]

Testing CohereForAI/aya-expanse-8b:   1%|▏         | 2/147 [00:05<07:06,  2.94s/it]

Testing CohereForAI/aya-expanse-8b:   2%|▏         | 3/147 [00:08<06:27,  2.69s/it]

Testing CohereForAI/aya-expanse-8b:   3%|▎         | 4/147 [00:11<06:39,  2.79s/it]

Testing CohereForAI/aya-expanse-8b:   3%|▎         | 5/147 [00:14<06:45,  2.86s/it]

Testing CohereForAI/aya-expanse-8b:   4%|▍         | 6/147 [00:17<06:48,  2.90s/it]

Testing CohereForAI/aya-expanse-8b:   5%|▍         | 7/147 [00:20<06:48,  2.92s/it]

Testing CohereForAI/aya-expanse-8b:   5%|▌         | 8/147 [00:23<06:48,  2.94s/it]

Testing CohereForAI/aya-expanse-8b:   6%|▌         | 9/147 [00:26<06:47,  2.95s/it]

Testing CohereForAI/aya-expanse-8b:   7%|▋         | 10/147 [00:29<06:44,  2.95s/it]

Testing CohereForAI/aya-expanse-8b:   7%|▋         | 11/147 [00:32<06:

Successfully cleared Disk and VRAM for CohereForAI/aya-expanse-8b
Testing complete. Results saved to /kaggle/working/results_noncontextual_events.csv


## Category: Arts

## Type: Non Contextual

In [12]:
prompts = pd.read_csv(non_contextual_prompt_files_path['Arts'])
entities = pd.read_csv(entites_files_path['Art'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [13]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

,culture,prompt,local_entity,western_entity
0,Punjabi,میوزیم وچ میں اک خوبصورت [MASK] دیکھی,نقاشی,سوان لیک
1,Punjabi,اسکول وچ بچوں نے [MASK] بنایا,کلامِ بابا فرید,پال بونین
2,Punjabi,بازار وچ [MASK] دے نال رنگ برنگے پینٹنگاں فروخ...,چنیوٹی وڈ ورک,واشنگٹن کراسنگ دی ڈیلاویئر
3,Punjabi,میلہ وچ ہر کوئی [MASK] دیکھ کے خوش ہویا,تلا ورک,لا ٹراویاٹا
4,Punjabi,آج شام میں گیلری وچ [MASK] دے بارے پڑھیا,سسی پنوں,دی اوڈیسی
...,...,...,...,...
106,Pashtun,هغې میاشتې په [MASK] باندې د یوې پیچلې نمونې پ...,د رحمان بابا شاعري,اپالاچین اسپرنگ
107,Pashtun,هنرمند د دودیز [MASK] د انځورولو لپاره طبیعي ر...,کاکړۍ,سنڈریلا
108,Pashtun,د واده پر مهال، نارینه وو یوه زوروره [MASK] نڅ...,رباب موسیقي,این آف گرین گیبلز
109,Pashtun,ځايي لرګي توږونکي د خپلو مفصلو [MASK] له امله ...,د غني خان شاعري,دی اوڈیسی


In [14]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_noncontextual_arts.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")


--- Loading model: google/gemma-2-2b ---


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Testing CohereForAI/aya-expanse-8b: 100%|██████████| 111/111 [05:23<00:00,  2.92s/it]


Successfully cleared Disk and VRAM for CohereForAI/aya-expanse-8b
Testing complete. Results saved to /kaggle/working/results_noncontextual_arts.csv


## Category: Literature

## Type: Non Contextual

In [15]:
prompts = pd.read_csv(non_contextual_prompt_files_path['Literature'])
entities = pd.read_csv(entites_files_path['Literature'])
western_entities = entities['Western']
western_entities.dropna(inplace=True)

In [16]:
# punjabi culture vs western culture

punjabi_entities = entities["Punjabi"]
punjabi_entities.dropna(inplace=True)
punjabi_prompts = prompts['Punjabi']
punjabi_prompts.dropna(inplace=True)

sample_size = np.tile(punjabi_prompts, 3).size
punjabi_df = pd.DataFrame({
    'culture': 'Punjabi',
    'prompt': np.tile(punjabi_prompts, 3),
    'local_entity': punjabi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# sindhi culture vs western culture

sindhi_entities = entities["Sindhi"]
sindhi_entities.dropna(inplace=True)
sindhi_prompts = prompts['Sindhi']
sindhi_prompts.dropna(inplace=True)

sample_size = np.tile(sindhi_prompts, 3).size
sindhi_df = pd.DataFrame({
    'culture': 'Sindhi',
    'prompt': np.tile(sindhi_prompts, 3),
    'local_entity': sindhi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# Balochi culture vs western culture

balochi_entities = entities["Balochi"]
balochi_entities.dropna(inplace=True)
balochi_prompts = prompts['Balochi']
balochi_prompts.dropna(inplace=True)

sample_size = np.tile(balochi_prompts, 3).size
balochi_df = pd.DataFrame({
    'culture': 'Balochi',
    'prompt': np.tile(balochi_prompts, 3),
    'local_entity': balochi_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

# pashtun culture vs western culture

pashtun_entities = entities["Pashtun"]
pashtun_entities.dropna(inplace=True)
pashtun_prompts = prompts['Pashtun']
pashtun_prompts.dropna(inplace=True)

sample_size = np.tile(pashtun_prompts, 3).size
pashtun_df = pd.DataFrame({
    'culture': 'Pashtun',
    'prompt': np.tile(pashtun_prompts, 3),
    'local_entity': pashtun_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True),
    'western_entity': western_entities.sample(sample_size, random_state=42, replace=True).reset_index(drop=True)
})

df = pd.concat([punjabi_df,sindhi_df, balochi_df, pashtun_df], axis=0, ignore_index=True)
df

,culture,prompt,local_entity,western_entity
0,Punjabi,اسکول وچ میں کلاس دے استاد توں [MASK] بارے سکھیا,ماہیا,سٹائر
1,Punjabi,کل میں اک خوبصورت [MASK] پڑھی,جنگ نامہ,چارلس بوڈلیئر
2,Punjabi,لائبریری وچ میں [MASK] دی کتاب لئی تلاش کیتی,سرجیت پاتر,فاسٹ
3,Punjabi,شاعری دی محفل وچ [MASK] بہت متاثر کن سی,مثنوی,پوسٹ ماڈرنزم
4,Punjabi,میرے دوست نے نواں [MASK] لکھیا,قادر یار,دانتے الیگیری
...,...,...,...,...
103,Pashtun,د [MASK] وزن او قافیه د هغې د ښکلا لپاره اړین دي.,مخزن الاسلام,رومانٹیسزم
104,Pashtun,پوهان هر کال د پښتو [MASK] د تکامل په اړه د بح...,رباعي,کامیڈی
105,Pashtun,هغه یو مشهور [MASK] ووایه چې د ننګ او غیرت په ...,احمد علي سائیں,پکاریسک
106,Pashtun,ماشومانو ته له ډیر کوچنيوالي څخه د [MASK] یادو...,ډاکټر اعظم اعظم,ایگزسٹنشلیزم


In [17]:
import torch
import pandas as pd
import numpy as np
import gc
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.nn.functional import softmax
from tqdm import tqdm

CACHE_DIR = "/kaggle/working/hf_cache"
OUTPUT_FILE = '/kaggle/working/results_noncontextual_literature.csv'

class CulturalBiasTester:
    def __init__(self, model_id, cache_dir):
        print(f"\n--- Loading model: {model_id} ---")
        self.model_id = model_id
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=cache_dir
        )
        
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            cache_dir=cache_dir
        )

    def ask_llm(self, instruction):
        inputs = self.tokenizer(instruction, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            output_tokens = self.model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(output_tokens[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        return response.strip().lower()

    def get_cloze_probabilities(self, prompt, local_word, western_word):
        clean_prompt = prompt.replace("[MASK]", "").strip()
        inputs = self.tokenizer(clean_prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits[0, -1, :]
        probs = softmax(logits, dim=-1)
        
        local_id = self.tokenizer.encode(local_word, add_special_tokens=False)[0]
        western_id = self.tokenizer.encode(western_word, add_special_tokens=False)[0]
        
        return {"local_p": probs[local_id].item(), "western_p": probs[western_id].item()}

    def evaluate_sentiment_self(self, text):
        prompt = f"Analyze the sentiment of this text. Answer with only one word: Positive, Negative, or Neutral.\nText: {text}\nAnswer:"
        answer = self.ask_llm(prompt)
        if "positive" in answer: return 1
        if "negative" in answer: return -1
        return 0

    def check_ner_self(self, text, entity):
        prompt = f"In the sentence '{text}', is the word '{entity}' a person's name? Answer with only Yes or No.\nAnswer:"
        answer = self.ask_llm(prompt)
        return 1 if "yes" in answer else 0


MODELS = [
    "google/gemma-2-2b", 
    "Qwen/Qwen2.5-7B-Instruct",
    "enstazao/Qalb-1.0-8B-Instruct",
    # "large-traversaal/Alif-1.0-8B-Instruct",
    "muhammadnoman76/Lughaat-1.0-8B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "CohereForAI/aya-expanse-8b",
    # "openai/gpt-oss-20b"
]

all_results = []

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

for m_id in MODELS:
    try:
        tester = CulturalBiasTester(m_id, CACHE_DIR)
        
        # 2. Run Inference
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Testing {m_id}"):
            probs = tester.get_cloze_probabilities(row['prompt'], row['local_entity'], row['western_entity'])
            sentence_local = row['prompt'].replace("[MASK]", row['local_entity'])
            sentence_western = row['prompt'].replace("[MASK]", row['western_entity'])
            
            sent_local = tester.evaluate_sentiment_self(sentence_local)
            sent_western = tester.evaluate_sentiment_self(sentence_western)
            ner_success = tester.check_ner_self(sentence_local, row['local_entity'])
            
            bias_ratio = probs['western_p'] / (probs['local_p'] + 1e-10)
            
            all_results.append({
                "model": m_id,
                "culture": row['culture'],
                "prompt": row['prompt'],
                "local_name": row['local_entity'],
                "western_name": row['western_entity'],
                "local_prob": probs['local_p'],
                "western_prob": probs['western_p'],
                "bias_ratio": round(bias_ratio, 4),
                "sentiment_diff": sent_local - sent_western,
                "ner_self_recognized": ner_success
            })

        # memory cleanup
        del tester
        gc.collect()
        torch.cuda.empty_cache()

        # wipe the hard cache
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
        print(f"Successfully cleared Disk and VRAM for {m_id}")

    except Exception as e:
        print(f"Error processing model {m_id}: {e}")
        if os.path.exists(CACHE_DIR):
            shutil.rmtree(CACHE_DIR)
            os.makedirs(CACHE_DIR)

final_df = pd.DataFrame(all_results)
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Testing complete. Results saved to {OUTPUT_FILE}")


--- Loading model: google/gemma-2-2b ---


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]



Testing CohereForAI/aya-expanse-8b:   0%|          | 0/108 [00:00<?, ?it/s]

Testing CohereForAI/aya-expanse-8b:   1%|          | 1/108 [00:03<05:22,  3.02s/it]

Testing CohereForAI/aya-expanse-8b:   2%|▏         | 2/108 [00:05<05:12,  2.95s/it]

Testing CohereForAI/aya-expanse-8b:   3%|▎         | 3/108 [00:08<05:09,  2.95s/it]

Testing CohereForAI/aya-expanse-8b:   4%|▎         | 4/108 [00:11<05:06,  2.95s/it]

Testing CohereForAI/aya-expanse-8b:   5%|▍         | 5/108 [00:14<05:03,  2.95s/it]

Testing CohereForAI/aya-expanse-8b:   6%|▌         | 6/108 [00:17<05:02,  2.97s/it]

Testing CohereForAI/aya-expanse-8b:   6%|▋         | 7/108 [00:20<04:58,  2.95s/it]

Testing CohereForAI/aya-expanse-8b:   7%|▋         | 8/108 [00:23<04:54,  2.95s/it]

Testing CohereForAI/aya-expanse-8b:   8%|▊         | 9/108 [00:26<04:52,  2.95s/it]

Testing CohereForAI/aya-expanse-8b:   9%|▉         | 10/108 [00:29<04:33,  2.79s/it]

Testing CohereForAI/aya-expanse-8b:  10%|█         | 11/108 [00:31<04:

Successfully cleared Disk and VRAM for CohereForAI/aya-expanse-8b
Testing complete. Results saved to /kaggle/working/results_noncontextual_literature.csv


In [ ]:
df = pd.read_csv('/kaggle/working/results_noncontextual_arts.csv')

In [ ]:
df

In [ ]:
import pandas as pd